In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# 1. Load Cleaned Dataset
df = pd.read_csv("cleaned_data.csv")


# Feature Engineering for Predictive Modeling
# Customer Preference Score (1-100) based on fabric technology, trend velocity, and price point
def calculate_preference(row):
  score = 50
  if row['Brand'] in ['UNIQLO'] and (
      'AIRism' in str(row['Category']) or 'HEATTECH' in str(row['Category'])
  ):
    score += 35  # High demand for proprietary functional utility
  elif row['Brand'] in ['Nike', 'adidas'] and (
      'Footwear' in str(row['Category']) or 'Jerseys' in str(row['Category'])
  ):
    score += 30  # High demand for athletic performance gear
  elif row['Brand'] == 'Zara' and 'Blazers' in str(row['Category']):
    score += 25  # High demand for fast fashion tailored wear
  elif row['Price_INR'] < 2000:
    score += 20  # High volume budget preference
  return min(score, 98)


df['Customer_Preference_Index'] = df.apply(calculate_preference, axis=1)

# One-Hot Encoding for Categorical Variables
df_encoded = pd.get_dummies(
    df[['Brand', 'Category', 'Price_INR', 'Customer_Preference_Index']],
    drop_first=True,
)

X = df_encoded.drop(columns=['Price_INR'])
y = df_encoded['Price_INR']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Model 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Metrics Calculation
lr_r2 = r2_score(y_test, y_pred_lr)
rf_r2 = r2_score(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("=== PREDICTIVE MODEL EVALUATION ===")
print(f"Linear Regression R2 Score : {lr_r2:.4f}")
print(f"Random Forest R2 Score     : {rf_r2:.4f}")
print(f"Random Forest RMSE         : ₹{rf_rmse:.2f}")

# Save full predictions to dataset for Power BI
df['Predicted_Price_INR'] = rf_model.predict(X)
df.to_csv("predicted_retail_data.csv", index=False)

# Chart 1: Actual vs Predicted Prices Plot
plt.figure(figsize=(9, 5))
plt.scatter(
    y_test,
    y_pred_rf,
    color='#2b6cb0',
    s=80,
    edgecolors='k',
    alpha=0.8,
    label='Predicted vs Actual',
)
plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    'r--',
    lw=2,
    label='Perfect Prediction Line',
)
plt.title(
    "Actual vs Predicted Retail Prices (Random Forest Model)",
    fontsize=13,
    fontweight='bold',
)
plt.xlabel("Actual Price in INR (₹)", fontsize=11, fontweight='bold')
plt.ylabel("Predicted Price in INR (₹)", fontsize=11, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig("week4_chart1_actual_vs_predicted.png", dpi=300)
plt.close()

# Chart 2: Predicted Customer Preference vs Price Point
plt.figure(figsize=(9, 5))
sns.scatterplot(
    x='Price_INR',
    y='Customer_Preference_Index',
    hue='Brand',
    data=df,
    s=110,
    palette='Set1',
)
plt.title(
    "Predictive Map: Customer Preference Index vs. Price Point",
    fontsize=13,
    fontweight='bold',
)
plt.xlabel("Price Point in INR (₹)", fontsize=11, fontweight='bold')
plt.ylabel("Customer Preference Index (1-100)", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("week4_chart2_preference_prediction.png", dpi=300)
plt.close()

print("\nWeek 4 Modeling Script Executed Successfully!")

=== PREDICTIVE MODEL EVALUATION ===
Linear Regression R2 Score : 0.1124
Random Forest R2 Score     : 0.4656
Random Forest RMSE         : ₹1519.58

Week 4 Modeling Script Executed Successfully!
